**Setup:**

1. **Create and activate a virtual environment** (from the project root):
   ```bash
   python3 -m venv .venv
   source .venv/bin/activate   # Windows: .venv\Scripts\activate
   ```
2. **Install dependencies:** `pip install -r requirements.txt`
3. **Create a `.env` file** in the project root with your OpenAI API key:
   ```
   OPENAI_API_KEY=your-key-here
   ```
4. **Select this environment** as the notebook kernel (`.venv` interpreter).

In [6]:
# Constants for test case generation
import os
from dotenv import load_dotenv

load_dotenv()
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
APP_NAME = "booking.com"
APP_PACKAGE_NAME = "com.booking"
APP_DESCRIPTION_BASIC = "Book your whole trip in one app."
APP_CONTEXT = """
Add app context here
"""

# Model used for BDD test generation (e.g. gpt-4o-mini, gpt-4o)
MODEL_NAME = os.environ.get("OPENAI_MODEL", "gpt-4.1")

In [4]:
# LLM client and helper for BDD test case generation
import base64
import json
import os
import re
from pathlib import Path
from openai import OpenAI

if not OPENAI_API_KEY :
    raise ValueError("Set OPENAI_API_KEY in the .env file or in your environment.")

client = OpenAI(api_key=OPENAI_API_KEY)

# Shared screenshots directory for sections 2 and 3 (place PNG/JPG etc. here)
SCREENSHOTS_DIR = Path("screenshots")

BDD_SCHEMA = """
Return a JSON array of strings. Each string is one complete BDD test scenario as a multiline block:
- Line 1: Given ... (starting state; e.g. "Given the user is on the <screen category> screen")
- Line 2: When ... (action)
- Line 3: Then ... (expected outcome)
Each array element = one full scenario. Output only valid JSON, no markdown or extra text.
"""


def screenshots_as_prompt_append(screenshot_dir: Path) -> str:
    """Load images from directory and return a string to append to the user prompt (base64 data URLs)."""
    if not screenshot_dir.exists() or not screenshot_dir.is_dir():
        return ""
    parts = []
    for ext in ("*.png", "*.jpg", "*.jpeg", "*.webp", "*.gif"):
        for path in sorted(screenshot_dir.glob(ext)):
            data = path.read_bytes()
            b64 = base64.standard_b64encode(data).decode("ascii")
            mime = "image/png" if path.suffix.lower() == ".png" else "image/jpeg"
            if path.suffix.lower() in (".webp", ".gif"):
                mime = f"image/{path.suffix.lower().lstrip('.')}"
            parts.append(f"[{path.name}] data:{mime};base64,{b64}")
    if not parts:
        return ""
    return "\n\nScreenshots (base64 data URLs for reference):\n" + "\n".join(parts)


def call_llm_for_bdd(system_prompt: str, user_prompt: str, model: str = None) -> list[str]:
    """Call OpenAI and parse response into list of BDD scenario strings (one multiline string per test)."""
    if model is None:
        model = MODEL_NAME
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.3,
    )
    text = response.choices[0].message.content.strip()
    # Strip markdown code block if present
    if "```json" in text:
        text = re.search(r"```(?:json)?\s*([\s\S]*?)```", text).group(1).strip()
    elif "```" in text:
        text = re.sub(r"^```\w*\s*", "", text).strip()
        text = re.sub(r"\s*```$", "", text).strip()
    data = json.loads(text)
    if isinstance(data, list) and data and isinstance(data[0], str):
        return data
    if isinstance(data, list) and data and isinstance(data[0], dict):
        out = []
        for f in data:
            for s in f.get("scenarios", []):
                steps = s.get("steps", [])
                lines = [f"{st.get('type', '')} {st.get('text', '')}".strip() for st in steps]
                out.append("\n".join(lines))
        return out
    return data if isinstance(data, list) else [str(data)]

### Sample test case generation


In [5]:
# Quick sample: call the LLM and print the result (list of multiline BDD strings)
system = f"""You are a QA engineer. Generate mobile app test cases in BDD format (Given/When/Then).
{BDD_SCHEMA}"""
user = f"""Generate 4-6 test scenarios for the mobile app "{APP_NAME}". Each scenario = one multiline string (Given ... When ... Then ...). Start from whatever screens make sense (login, home, etc.). Output only a JSON array of strings."""

sample_output = call_llm_for_bdd(system, user)
for i, scenario in enumerate(sample_output, 1):
    print(f"--- Test {i} ---\n{scenario}\n")

--- Test 1 ---
Given the user is on the login screen
When the user enters valid credentials and taps the login button
Then the user should be redirected to the home screen

--- Test 2 ---
Given the user is on the home screen
When the user enters a city name in the search bar and taps the search button
Then the app should display a list of available accommodations in that city

--- Test 3 ---
Given the user is viewing the list of search results
When the user selects a hotel from the list
Then the app should display the hotel's details page

--- Test 4 ---
Given the user is on a hotel details page
When the user taps the 'Book Now' button and completes the booking process
Then the app should display a booking confirmation screen

--- Test 5 ---
Given the user is on the home screen
When the user taps the 'My Bookings' tab
Then the app should display a list of the user's current and past bookings

--- Test 6 ---
Given the user is on the booking confirmation screen
When the user taps the 'Sh

## 1. Simple test case generation (app name only)

Generates BDD-style test cases using only the app name. Saves to `test_cases_simple/`.

In [ ]:
import json
from pathlib import Path

OUTPUT_DIR_SIMPLE = Path("test_cases_simple")
OUTPUT_DIR_SIMPLE.mkdir(exist_ok=True)


def generate_test_cases_simple(app_name: str) -> list[str]:
    """
    Generate BDD test cases based on app name only.
    Returns list of multiline strings, each string = one full BDD scenario (Given/When/Then).
    """
    system = f"""You are a QA engineer. Generate mobile app test cases in BDD format (Given/When/Then).
{BDD_SCHEMA}"""
    user = f"""Generate as many test scenarios as possible for the mobile app "{app_name}({APP_DESCRIPTION_BASIC})". Each scenario = one multiline string (Given ... When ... Then ...). Start from whatever screens make sense. Output only a JSON array of strings."""
    return call_llm_for_bdd(system, user)


# Generate and save
cases_simple = generate_test_cases_simple(APP_NAME)
out_file = OUTPUT_DIR_SIMPLE / f"{APP_NAME.replace(' ', '_')}_test_cases.json"
with open(out_file, "w") as f:
    json.dump(cases_simple, f, indent=2)
print(f"Saved {len(cases_simple)} scenario(s) to {out_file}")

Saved 30 scenario(s) to test_cases_prompt_engineering_v1/booking.com_test_cases.json


## 2. Test case generation with app name and context

Generates BDD test cases using app name and app context (domain, target platform, etc.). Optionally ingests **screenshots** from `screenshots/` (PNG, JPG, etc.) to improve test coverage. Saves to `test_cases_with_context/`.

In [ ]:
OUTPUT_DIR_CONTEXT = Path("test_cases_with_context")
OUTPUT_DIR_CONTEXT.mkdir(exist_ok=True)

def generate_test_cases_with_context(
    app_name: str,
    app_context: str,
    screenshot_dir: Path | None = None,
) -> list[str]:
    """
    Generate BDD test cases using app name and context (domain, platform, etc.).
    If screenshot_dir is provided and contains images, their base64 data URLs are appended to the prompt.
    Returns list of multiline strings, each = one full BDD scenario.
    """
    system = f"""You are a QA engineer. Generate mobile app test cases in BDD format (Given/When/Then) tailored to the app's domain and context.
{BDD_SCHEMA}"""
    user = f"""App: "{app_name}"

Context:
{app_context.strip()}

Generate as many test scenarios as possible. Each = one multiline string (Given ... When ... Then ...). Start from whatever screens fit. Output only a JSON array of strings."""
    if screenshot_dir and screenshot_dir.exists():
        append_str = screenshots_as_prompt_append(screenshot_dir)
        if append_str:
            user += append_str
            user += "\n\nUse the provided screenshots to align scenarios with the actual app screens and flows."
    return call_llm_for_bdd(system, user)


# Generate and save (ingests screenshots from screenshots/ if present)
cases_context = generate_test_cases_with_context(APP_NAME, APP_CONTEXT, screenshot_dir=SCREENSHOTS_DIR)
out_file = OUTPUT_DIR_CONTEXT / f"{APP_NAME.replace(' ', '_')}_test_cases.json"
with open(out_file, "w") as f:
    json.dump(cases_context, f, indent=2)
print(f"Saved {len(cases_context)} scenario(s) to {out_file}")

## 3. Test case generation from app graph (any JSON structure)

Reads a JSON file from `graphs/MY_APP_graph.json` (any structure—nodes/edges or otherwise), stringifies it, and adds it to the prompt for BDD test generation. Optionally ingests **screenshots** from `screenshots/`. Saves to `test_cases_from_graph/`.

In [10]:
# Path to the graph JSON (any structure)
APP_GRAPH_JSON_PATH = Path("graphs/MY_APP_graph.json")

OUTPUT_DIR_GRAPH = Path("test_cases_from_graph")
OUTPUT_DIR_GRAPH.mkdir(exist_ok=True)


def load_app_graph(graph_path: Path) -> dict | list:
    """Load JSON from file. Returns parsed object (dict or list) or empty dict if missing/invalid."""
    if not graph_path.exists():
        return {}
    try:
        with open(graph_path) as f:
            return json.load(f)
    except (json.JSONDecodeError, OSError):
        return {}


def generate_test_cases_from_graph(
    app_name: str,
    app_context: str,
    graph: dict | list,
    screenshot_dir: Path | None = None,
) -> list[str]:
    """
    Generate BDD test cases from app graph. Graph can be any JSON structure—it is stringified and added to the prompt.
    If screenshot_dir is provided and contains images, their base64 data URLs are appended to the prompt.
    Returns list of multiline strings, each = one full BDD scenario.
    """
    
    system = f"""You are a QA engineer. Given an app graph and context, generate BDD test cases that cover screens and flows.
{BDD_SCHEMA}"""

    context_str = f"App context: {app_context.strip()}"
    graph_str = json.dumps(graph, indent=2) if graph else "{}"
    user = f"""App: "{app_name}"

    App context:
    {context_str}

    App graph (any structure):
    {graph_str}

    Generate as many BDD scenarios as possible for all user flows possible in the app. Each scenario = one multiline string (Given ... When ... Then ...). Output only a JSON array of strings."""
    if screenshot_dir and screenshot_dir.exists():
        append_str = screenshots_as_prompt_append(screenshot_dir)
        if append_str:
            user += append_str
            user += "\n\nUse the provided screenshots (UI/code) to align scenarios with the actual screens and flows."
    return call_llm_for_bdd(system, user)


# Generate from graph file (empty dict if file missing)
graph = load_app_graph(APP_GRAPH_JSON_PATH)
cases_graph = generate_test_cases_from_graph(
    APP_NAME, APP_CONTEXT, graph, screenshot_dir=SCREENSHOTS_DIR
)
out_file = OUTPUT_DIR_GRAPH / f"{APP_NAME.replace(' ', '_')}_graph_test_cases.json"
with open(out_file, "w") as f:
    json.dump(cases_graph, f, indent=2)
print(f"Saved {len(cases_graph)} scenario(s) to {out_file}")

Saved 9 scenario(s) to test_cases_from_graph/booking.com_graph_test_cases.json
